# NboDriver Test and Demo Notebook

Development checks for the `NboDriver` implementation in VeloxChem.
This notebook demonstrates formaldehyde reports, multi-molecule NBO count sanity checks, and first-pass Lewis/resonance score-weight diagnostics.

## 1. Imports

In [5]:
import importlib.util
import sys
from pathlib import Path

import numpy as np
import veloxchem as vlx

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "src/pymodule/nbodriver.py").exists():
    repo_root = repo_root.parent
if not (repo_root / "src/pymodule/nbodriver.py").exists():
    raise RuntimeError("Could not locate the VeloxChem repository root")

def load_local_driver_module(module_name, relative_path):
    module_path = repo_root / relative_path
    spec = importlib.util.spec_from_file_location(module_name, module_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module

analyzer_module = load_local_driver_module(
    "veloxchem.orbitalanalyzerdriver",
    "src/pymodule/orbitalanalyzerdriver.py",
)
nbodriver_module = load_local_driver_module(
    "veloxchem.nbodriver",
    "src/pymodule/nbodriver.py",
)
vlx.NboDriver = nbodriver_module.NboDriver

print("VeloxChem version:", vlx.__version__)
print("Repository root:", repo_root)
print("NboDriver module:", nbodriver_module.__file__)
print("OrbitalAnalyzer module:", analyzer_module.__file__)


VeloxChem version: 1.0rc4
Repository root: /home/linares/app/VeloxChem
NboDriver module: /home/linares/app/VeloxChem/src/pymodule/nbodriver.py
OrbitalAnalyzer module: /home/linares/app/VeloxChem/src/pymodule/orbitalanalyzerdriver.py


## 2. Build formaldehyde (H2CO) and run SCF

In [ ]:
# Formaldehyde from Z-matrix (Angstrom / degrees)
z_matrix = """
C
O  1  1.21672286
H  1  1.10137241  2  122.73666566
H  1  1.10137241  2  122.73666566  3  180.0
"""

def _unit(vec):
    nrm = np.linalg.norm(vec)
    if nrm < 1.0e-12:
        raise ValueError("Encountered near-zero vector in Z-matrix conversion")
    return vec / nrm

def _perp_to(vec):
    trial = np.array([1.0, 0.0, 0.0])
    if abs(np.dot(_unit(vec), trial)) > 0.9:
        trial = np.array([0.0, 1.0, 0.0])
    return _unit(np.cross(vec, trial))

def zmat_to_xyz(zmat_text):
    lines = [ln.strip() for ln in zmat_text.strip().splitlines() if ln.strip()]
    entries = []
    for ln in lines:
        parts = ln.split()
        e = {'el': parts[0]}
        if len(parts) >= 3:
            e['a'] = int(parts[1]) - 1
            e['r'] = float(parts[2])
        if len(parts) >= 5:
            e['b'] = int(parts[3]) - 1
            e['ang'] = np.deg2rad(float(parts[4]))
        if len(parts) >= 7:
            e['c'] = int(parts[5]) - 1
            e['dih'] = np.deg2rad(float(parts[6]))
        entries.append(e)

    coords = []
    symbols = []

    for i, e in enumerate(entries):
        symbols.append(e['el'])

        if i == 0:
            coords.append(np.array([0.0, 0.0, 0.0]))
            continue

        if i == 1:
            ra = coords[e['a']]
            coords.append(ra + np.array([0.0, 0.0, e['r']]))
            continue

        if i == 2:
            ra = coords[e['a']]
            rb = coords[e['b']]
            ab = _unit(rb - ra)  # a -> b
            p1 = _perp_to(ab)
            local = np.cos(e['ang']) * ab + np.sin(e['ang']) * p1
            coords.append(ra + e['r'] * local)
            continue

        ra = coords[e['a']]
        rb = coords[e['b']]
        rc = coords[e['c']]

        ab = _unit(rb - ra)      # a -> b
        bc = _unit(rc - rb)      # b -> c

        n = np.cross(ab, bc)
        if np.linalg.norm(n) < 1.0e-10:
            n = _perp_to(ab)
        else:
            n = _unit(n)

        p1 = _unit(np.cross(n, ab))
        p2 = n

        local = (
            np.cos(e['ang']) * ab
            + np.sin(e['ang']) * (np.cos(e['dih']) * p1 + np.sin(e['dih']) * p2)
        )
        coords.append(ra + e['r'] * local)

    return symbols, np.array(coords)

symbols, xyz = zmat_to_xyz(z_matrix)
mol_str = "\n".join(
    f"{sym:2s}  {x: .10f}  {y: .10f}  {z: .10f}"
    for sym, (x, y, z) in zip(symbols, xyz)
 )

molecule = vlx.Molecule.read_str(mol_str)

print(mol_str)


C    0.0000000000   0.0000000000   0.0000000000
O    0.0000000000   0.0000000000   1.2167228600
H    0.0000000000   0.9264358023  -0.5955987657
H   -0.0000000000  -0.9264358023  -0.5955987657


In [ ]:

basis    = vlx.MolecularBasis.read(molecule, "STO-3G")

molecule.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
# Run restricted SCF
scf_drv = vlx.ScfRestrictedDriver()
scf_drv.xcfun_label = "hf"
scf_results = scf_drv.compute(molecule, basis)

                                                                                                                          
                                            Self Consistent Field Driver Setup                                            
                                                                                                                          
                   Wave Function Model             : Spin-Restricted Hartree-Fock                                         
                   Initial Guess Model             : Superposition of Atomic Densities                                    
                   Convergence Accelerator         : Two Level Direct Inversion of Iterative Subspace                     
                   Max. Number of Iterations       : 50                                                                   
                   Max. Number of Error Vectors    : 10                                                                   
                

## 3. Run NBO analysis

In [ ]:
nbo_drv = vlx.NboDriver()
nbo_results = nbo_drv.compute(molecule, basis, scf_drv.mol_orbs)

                                                                                                                          
A Summary of Natural Population Analysis:

  Atom #  Natural Charge        Core     Valence     Rydberg       Total
--------------------------------------------------------------------------
 C  1        +0.08308     2.00000     3.91692     0.00000     5.91692
 O  2        -0.11587     2.00000     6.11587     0.00000     8.11587
 H  3        +0.01640     0.00000     0.98360     0.00000     0.98360
 H  4        +0.01640     0.00000     0.98360     0.00000     0.98360
 * Total *        +0.00000     4.00000    12.00000     0.00000    16.00000
                                                                                                                          
Natural Bond Orbital (NBO) Primary Summary
Lewis assignment from generated NBO candidates.
Available layers: NAO/NPA, MO-in-NAO analysis, candidates, alternatives, diagnostics, and NRA/NRT.
Report order: CR, 

## 3b. Shared OrbitalAnalyzer payload check

`NboDriver` now receives the NAO/NPA, MO-in-NAO, and candidate layer from the shared `OrbitalAnalyzer` payload. This cell checks that the NBO result exposes that payload and that the candidate table is the same object consumed by the NBO report layer.

In [ ]:
import numpy as np

analysis = nbo_results.get("orbital_analysis")
assert analysis is not None, "NboDriver did not return the shared OrbitalAnalyzer payload"
assert np.allclose(analysis.nao_data.transform, nbo_results["nao_transform"])
assert len(analysis.orbital_candidates) == len(nbo_results["nbo_candidates"])

candidate_labels = [
    f"{c['type']}_{c.get('subtype', '')}_{c['index']}".replace("__", "_")
    for c in analysis.orbital_candidates
]
print("OrbitalAnalyzer candidate count:", len(candidate_labels))
print("Candidate labels:", candidate_labels)
print("Candidate types:", sorted({c["type"] for c in analysis.orbital_candidates}))

## 4. Natural Population Analysis (API-level report)

Use the NBO API report formatter directly.
You can switch level with: `none`, `summary`, `standard`, `full`.

In [ ]:
# API-level NPA report
nbo_drv.npa_report(level="full")

NATURAL POPULATIONS: Natural atomic orbital occupancies

 NAO  Atom #     l   Type(AO)     Occupancy
--------------------------------------------------------------------------
   1  C  1     s   Cor(1s)        2.00000
   2  C  1     s   Val(2s)        1.08832
   7  C  1     p   Val(2p)        1.03547
   9  C  1     p   Val(2p)        0.91641
  11  C  1     p   Val(2p)        0.87672
   3  O  2     s   Cor(1s)        2.00000
   4  O  2     s   Val(2s)        1.99755
   8  O  2     p   Val(2p)        1.91427
  10  O  2     p   Val(2p)        1.12046
  12  O  2     p   Val(2p)        1.08359
   5  H  3     s   Val(1s)        0.98360
   6  H  4     s   Val(1s)        0.98360

A Summary of Natural Population Analysis:

  Atom #  Natural Charge        Core     Valence     Rydberg       Total
--------------------------------------------------------------------------
 C  1        +0.08308     2.00000     3.91692     0.00000     5.91692
 O  2        -0.11587     2.00000     6.11587     0.00000 

'NATURAL POPULATIONS: Natural atomic orbital occupancies\n\n NAO  Atom #     l   Type(AO)     Occupancy\n--------------------------------------------------------------------------\n   1  C  1     s   Cor(1s)        2.00000\n   2  C  1     s   Val(2s)        1.08832\n   7  C  1     p   Val(2p)        1.03547\n   9  C  1     p   Val(2p)        0.91641\n  11  C  1     p   Val(2p)        0.87672\n   3  O  2     s   Cor(1s)        2.00000\n   4  O  2     s   Val(2s)        1.99755\n   8  O  2     p   Val(2p)        1.91427\n  10  O  2     p   Val(2p)        1.12046\n  12  O  2     p   Val(2p)        1.08359\n   5  H  3     s   Val(1s)        0.98360\n   6  H  4     s   Val(1s)        0.98360\n\nA Summary of Natural Population Analysis:\n\n  Atom #  Natural Charge        Core     Valence     Rydberg       Total\n--------------------------------------------------------------------------\n C  1        +0.08308     2.00000     3.91692     0.00000     5.91692\n O  2        -0.11587     2.00000  

## 4b. Molecular orbital composition in the NAO basis

This reports the canonical MO expansion in the orthonormal NAO basis built by `NboDriver`.

In [ ]:
# API-level MO-in-NAO report
nbo_drv.mo_report(level="occupied")

MO ANALYSIS: Molecular orbital composition in the NAO basis
Weights are squared NAO expansion coefficients and sum to one per MO.

Spin block: restricted
--------------------------------------------------------------------------------------------------------
  MO      Occ    Energy/a.u.  Dominant atoms             Dominant NAOs
--------------------------------------------------------------------------------------------------------
   1   2.0000     -20.312713  O2:100.0%                  3(O2s):98.1%, 4(O2s): 1.8%
   2   2.0000     -11.125070  C1: 99.9%                  1(C1s):99.8%
   3   2.0000      -1.337439  O2: 67.2%, C1: 30.6%, H3:  1.1% 10(O2p):35.7%, 4(O2s):30.6%, 11(C1p):26.2%, 2(C1s): 4.3%, 5(H3s): 1.1%, 6(H4s): 1.1%
   4   2.0000      -0.807753  C1: 46.5%, H3: 18.3%, H4: 18.3% 2(C1s):46.2%, 5(H3s):18.3%, 6(H4s):18.3%, 4(O2s):16.6%
   5   2.0000      -0.632912  C1: 46.3%, O2: 25.1%, H3: 14.3% 7(C1p):46.3%, 8(O2p):25.1%, 5(H3s):14.3%, 6(H4s):14.3%
   6   2.0000      -0.545533  

'MO ANALYSIS: Molecular orbital composition in the NAO basis\n========================================================================================================\nWeights are squared NAO expansion coefficients and sum to one per MO.\n\nSpin block: restricted\n--------------------------------------------------------------------------------------------------------\n  MO      Occ    Energy/a.u.  Dominant atoms             Dominant NAOs\n--------------------------------------------------------------------------------------------------------\n   1   2.0000     -20.312713  O2:100.0%                  3(O2s):98.1%, 4(O2s): 1.8%\n   2   2.0000     -11.125070  C1: 99.9%                  1(C1s):99.8%\n   3   2.0000      -1.337439  O2: 67.2%, C1: 30.6%, H3:  1.1% 10(O2p):35.7%, 4(O2s):30.6%, 11(C1p):26.2%, 2(C1s): 4.3%, 5(H3s): 1.1%, 6(H4s): 1.1%\n   4   2.0000      -0.807753  C1: 46.5%, H3: 18.3%, H4: 18.3% 2(C1s):46.2%, 5(H3s):18.3%, 6(H4s):18.3%, 4(O2s):16.6%\n   5   2.0000      -0.632912 

## 5. Natural Bond Orbital list (API-level report)

Use the NBO API report function directly.
You can switch level with: `none`, `summary`, `full`.

In [ ]:
# API-level NBO report
nbo_drv.nbo_report(level="full")

Natural Bond Orbital (NBO) Primary Summary
Lewis assignment from generated NBO candidates.
Available layers: NAO/NPA, MO-in-NAO analysis, candidates, alternatives, diagnostics, and NRA/NRT.
Report order: CR, BD(sigma), BD(pi), LP, RY, then antibonding/other candidates.

Primary counts: none
Candidate pool: none
Selected pairs: 0/8  occupation sum=0.00000  score=-86.00000
Primary alternative rank=1  Score weight=1.00000

 NBO Type       Atoms                 Occ  Polarization             Source
--------------------------------------------------------------------------------------------------------

Lewis/resonance alternatives
--------------------------------------------------------------------------------------------------------
Rank Score weight        Score   Pairs  Sigma/Pi Pi bonds
--------------------------------------------------------------------------------------------------------
   1      1.00000    -86.00000       0/8      0/0    none

Alternative notes
- Forbidden bonds wer

'Natural Bond Orbital (NBO) Primary Summary\n========================================================================================================\nLewis assignment from generated NBO candidates.\nAvailable layers: NAO/NPA, MO-in-NAO analysis, candidates, alternatives, diagnostics, and NRA/NRT.\nReport order: CR, BD(sigma), BD(pi), LP, RY, then antibonding/other candidates.\n\nPrimary counts: none\nCandidate pool: none\nSelected pairs: 0/8  occupation sum=0.00000  score=-86.00000\nPrimary alternative rank=1  Score weight=1.00000\nWarning: Forbidden bonds were excluded from primary assignment.\n\n NBO Type       Atoms                 Occ  Polarization             Source\n--------------------------------------------------------------------------------------------------------\n\nLewis/resonance alternatives\n--------------------------------------------------------------------------------------------------------\nRank Score weight        Score   Pairs  Sigma/Pi Pi bonds\n---------------

In [ ]:
# 6. Multi-molecule NBO sanity check (counts of BD / LP / CR)
import veloxchem as vlx

molecule_xyz = {
    "water": """
O   0.000000   0.000000   0.000000
H   0.758602   0.000000   0.504284
H  -0.758602   0.000000   0.504284
""",
    "methane": """
C   0.000000   0.000000   0.000000
H   0.629118   0.629118   0.629118
H  -0.629118  -0.629118   0.629118
H  -0.629118   0.629118  -0.629118
H   0.629118  -0.629118  -0.629118
""",
    "ethylene": """
C  -0.669500   0.000000   0.000000
C   0.669500   0.000000   0.000000
H  -1.232100   0.923000   0.000000
H  -1.232100  -0.923000   0.000000
H   1.232100   0.923000   0.000000
H   1.232100  -0.923000   0.000000
""",
    "benzene": """
C   1.396792   0.000000   0.000000
C   0.698396   1.209951   0.000000
C  -0.698396   1.209951   0.000000
C  -1.396792   0.000000   0.000000
C  -0.698396  -1.209951   0.000000
C   0.698396  -1.209951   0.000000
H   2.490290   0.000000   0.000000
H   1.245145   2.156660   0.000000
H  -1.245145   2.156660   0.000000
H  -2.490290   0.000000   0.000000
H  -1.245145  -2.156660   0.000000
H   1.245145  -2.156660   0.000000
""",
}

# Reference Lewis-like expectations for restricted closed-shell molecules.
# (BD = bonding NBO count, LP = lone-pair count, CR = core-like 1-center occupied)
# Benzene has 15 BD candidates in this representation: 6 C-H sigma, 6 C-C sigma, and 3 C-C pi.
expected = {
    "water":    {"BD": 2,  "LP": 2, "CR": 1},
    "methane":  {"BD": 4,  "LP": 0, "CR": 1},
    "ethylene": {"BD": 6,  "LP": 0, "CR": 2},
    "benzene":  {"BD": 15, "LP": 0, "CR": 6},
}

def nbo_type_counts(nbo_results):
    counts = {"CR": 0, "BD": 0, "LP": 0, "BD*": 0, "RY": 0}
    for item in nbo_results["nbo_list"]:
        t = item["type"]
        counts[t] = counts.get(t, 0) + 1
    return counts

print("NBO sanity check (HF/STO-3G)")
print("=" * 92)
print(f"{'Molecule':<10} {'BD(obs/exp)':<14} {'LP(obs/exp)':<14} {'CR(obs/exp)':<14} {'BD*':<6} {'RY':<6} {'Status'}")
print("-" * 92)

total_cases = 0
pass_cases = 0
check_cases = 0
check_names = []
all_counts = {}

for name, xyz in molecule_xyz.items():
    mol = vlx.Molecule.read_str(xyz)
    bas = vlx.MolecularBasis.read(mol, "sto-3g")

    scf = vlx.ScfRestrictedDriver()
    scf.xcfun_label = "hf"
    _ = scf.compute(mol, bas)

    nbo = vlx.NboDriver()
    nbo.verbose = False
    res = nbo.compute(mol, bas, scf.mol_orbs)
    obs = nbo_type_counts(res)
    all_counts[name] = dict(obs)

    exp = expected[name]
    ok = (obs["BD"] == exp["BD"]) and (obs["LP"] == exp["LP"]) and (obs["CR"] == exp["CR"])

    status = "PASS" if ok else "CHECK"

    total_cases += 1
    if ok:
        pass_cases += 1
    else:
        check_cases += 1
        check_names.append(name)

    print(
        f"{name:<10} "
        f"{obs['BD']:>2}/{exp['BD']:<10} "
        f"{obs['LP']:>2}/{exp['LP']:<10} "
        f"{obs['CR']:>2}/{exp['CR']:<10} "
        f"{obs['BD*']:<6d} "
        f"{obs['RY']:<6d} "
        f"{status}"
    )

    print()
    print(f"--- {name}: NBO report (full) ---")
    print(nbo.format_nbo_report(mol, res, level="full"))
    print("-" * 92)

print("=" * 92)
print("Note: BD*/RY are informative only here; pass/fail checks BD, LP, and CR counts.")
print()
print("Final summary:")
print(f"  Total molecules : {total_cases}")
print(f"  PASS            : {pass_cases}")
print(f"  CHECK           : {check_cases}")
if check_names:
    print("  CHECK molecules : " + ", ".join(check_names))
else:
    print("  CHECK molecules : none")

print()
print("All NBO counts by molecule:")
print("-" * 64)
print(f"{'Molecule':<10} {'CR':>4} {'BD':>4} {'LP':>4} {'BD*':>5} {'RY':>4}")
print("-" * 64)
for name in molecule_xyz:
    c = all_counts[name]
    print(f"{name:<10} {c['CR']:>4d} {c['BD']:>4d} {c['LP']:>4d} {c['BD*']:>5d} {c['RY']:>4d}")
print("-" * 64)

NBO sanity check (HF/STO-3G)
Molecule   BD(obs/exp)    LP(obs/exp)    CR(obs/exp)    BD*    RY     Status
--------------------------------------------------------------------------------------------
                                                                                                                          
                                            Self Consistent Field Driver Setup                                            
                                                                                                                          
                   Wave Function Model             : Spin-Restricted Hartree-Fock                                         
                   Initial Guess Model             : Superposition of Atomic Densities                                    
                   Convergence Accelerator         : Two Level Direct Inversion of Iterative Subspace                     
                   Max. Number of Iterations       : 50        

## 6b. Allyl cation, radical, and anion resonance score-weight check

This checks the two classical allyl Lewis structures, `C1=C2-C3` and `C1-C2=C3`, and the terminal `C1...C3` pi structure allowed through `allowed_pi_bonds`. The printed weights are current score/ranking weights, not physical NRA or VB weights.

In [ ]:
# Allyl resonance score-weight check
import veloxchem as vlx

allyl_xyz = """
C  -1.350000   0.000000   0.000000
C   0.000000   0.000000   0.000000
C   1.350000   0.000000   0.000000
H  -1.950000   0.900000   0.000000
H  -1.950000  -0.900000   0.000000
H   0.000000   1.080000   0.000000
H   1.950000   0.900000   0.000000
H   1.950000  -0.900000   0.000000
"""

# Two classical allyl Lewis pi structures plus the terminal non-sigma pi structure.
allyl_pi_structures = {
    "C1=C2-C3": (1, 2),
    "C1-C2=C3": (2, 3),
    "C1...C3": (1, 3),
}

allyl_cases = {
    "allyl cation":  {"charge": +1, "multiplicity": 1},
    "allyl radical": {"charge":  0, "multiplicity": 2},
    "allyl anion":   {"charge": -1, "multiplicity": 1},
}

def build_allyl_molecule(charge, multiplicity):
    mol = vlx.Molecule.read_str(allyl_xyz)
    mol.set_charge(charge)
    mol.set_multiplicity(multiplicity)
    return mol

def run_allyl_scf(mol, basis):
    if mol.get_multiplicity() == 1:
        scf = vlx.ScfRestrictedDriver()
    else:
        scf = vlx.ScfUnrestrictedDriver()
    scf.xcfun_label = "hf"
    _ = scf.compute(mol, basis)
    return scf

def pair_key(pair):
    return tuple(sorted(int(atom) for atom in pair))

def alternative_weight_map(res):
    weights = {pair_key(pair): 0.0 for pair in allyl_pi_structures.values()}
    scores = {pair_key(pair): None for pair in allyl_pi_structures.values()}
    for alt in res.get("alternatives", []):
        pi_bonds = [pair_key(pair) for pair in alt.get("pi_bonds", [])]
        if len(pi_bonds) != 1:
            continue
        pair = pi_bonds[0]
        if pair in weights:
            weights[pair] += float(alt.get("weight", 0.0))
            scores[pair] = float(alt.get("score", 0.0))
    return weights, scores

print("Allyl resonance score-weight check (HF/STO-3G)")
print("=" * 88)
print(f"{'Case':<15} {'Charge':>6} {'Mult':>4}  {'C1=C2-C3':>12} {'C1-C2=C3':>12} {'C1...C3':>12}")
print("-" * 88)

allyl_results = {}
for case_name, spec in allyl_cases.items():
    mol = build_allyl_molecule(spec["charge"], spec["multiplicity"])
    bas = vlx.MolecularBasis.read(mol, "def2-svp")
    scf = run_allyl_scf(mol, bas)

    nbo = vlx.NboDriver()
    nbo.verbose = False
    constraints = {
        # Allow all three pi choices. The terminal C1-C3 pi candidate has no sigma bond.
        "allowed_pi_bonds": list(allyl_pi_structures.values()),
    }
    options = {
        "include_diagnostics": True,
        "max_alternatives": 12,
        "pi_min_occupation": 0.05,
        "conjugated_pi_max_path": 2,
    }
    res = nbo.compute(mol, bas, scf.mol_orbs, constraints=constraints, options=options)
    weights, scores = alternative_weight_map(res)
    allyl_results[case_name] = {"results": res, "weights": weights, "scores": scores}

    w12 = weights[pair_key((1, 2))]
    w23 = weights[pair_key((2, 3))]
    w13 = weights[pair_key((1, 3))]
    print(
        f"{case_name:<15} {spec['charge']:>+6d} {spec['multiplicity']:>4d}  "
        f"{100*w12:>11.2f}% {100*w23:>11.2f}% {100*w13:>11.2f}%"
    )

print("=" * 88)
print("Detailed alternatives")
print("-" * 88)
for case_name, data in allyl_results.items():
    print(f"\n{case_name}")
    for alt in data["results"].get("alternatives", []):
        pi_text = ", ".join(f"{i}-{j}" for i, j in alt.get("pi_bonds", [])) or "none"
        print(
            f"  rank={alt.get('rank', 0):>2} "
            f"score_weight={100*alt.get('weight', 0.0):>7.2f}% "
            f"score={alt.get('score', 0.0):>10.5f} "
            f"pi={pi_text}"
        )

print("\nNote: allyl radical is open-shell; current Lewis enumeration is still pair-based, so treat radical score weights as diagnostic only.")

Allyl resonance score-weight check (HF/STO-3G)
Case            Charge Mult      C1=C2-C3     C1-C2=C3      C1...C3
----------------------------------------------------------------------------------------
                                                                                                                          
                                            Self Consistent Field Driver Setup                                            
                                                                                                                          
                   Wave Function Model             : Spin-Restricted Hartree-Fock                                         
                   Initial Guess Model             : Superposition of Atomic Densities                                    
                   Convergence Accelerator         : Two Level Direct Inversion of Iterative Subspace                     
                   Max. Number of Iterations       : 50   

## 6c. Benzene Kekule Lewis-structure score weights

This checks the two classical benzene Kekule pi structures. First the two Kekule patterns are allowed simultaneously so the automatic Lewis/resonance enumerator can assign score/ranking weights. Then each Kekule structure is forced with `required_pi_bonds` to verify that user-specified Lewis structures can be selected explicitly.

In [3]:
# Benzene Kekule and Dewar Lewis-structure score weights
import veloxchem as vlx

benzene_xyz = """
C   1.396792   0.000000   0.000000
C   0.698396   1.209951   0.000000
C  -0.698396   1.209951   0.000000
C  -1.396792   0.000000   0.000000
C  -0.698396  -1.209951   0.000000
C   0.698396  -1.209951   0.000000
H   2.490290   0.000000   0.000000
H   1.245145   2.156660   0.000000
H  -1.245145   2.156660   0.000000
H  -2.490290   0.000000   0.000000
H  -1.245145  -2.156660   0.000000
H   1.245145  -2.156660   0.000000
"""

benzene_structures = {
    # Two classical Kekule structures.
    "Kekule A (1-2,3-4,5-6)": [(1, 2), (3, 4), (5, 6)],
    "Kekule B (2-3,4-5,6-1)": [(2, 3), (4, 5), (6, 1)],

    # Three Dewar structures: one para/transannular pi bond plus two ring pi bonds.
    "Dewar A (1-4,2-3,5-6)": [(1, 4), (2, 3), (5, 6)],
    "Dewar B (2-5,3-4,6-1)": [(2, 5), (3, 4), (6, 1)],
    "Dewar C (3-6,4-5,1-2)": [(3, 6), (4, 5), (1, 2)],
}

ring_pi_bonds = [(1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 1)]
dewar_para_pi_bonds = [(1, 4), (2, 5), (3, 6)]
all_benzene_pi_bonds = ring_pi_bonds + dewar_para_pi_bonds

mol_benzene = vlx.Molecule.read_str(benzene_xyz)
bas_benzene = vlx.MolecularBasis.read(mol_benzene, "def2-svp")

scf_benzene = vlx.ScfRestrictedDriver()
scf_benzene.xcfun_label = "hf"
_ = scf_benzene.compute(mol_benzene, bas_benzene)

options_benzene = {
    "include_diagnostics": True,
    "max_alternatives": 24,
    "pi_min_occupation": 0.05,
    "conjugated_pi_max_path": 2,
}

def normalize_pi_set(pi_bonds):
    return tuple(sorted(tuple(sorted(pair)) for pair in pi_bonds))

def benzene_structure_weight_map(res):
    weights = {name: 0.0 for name in benzene_structures}
    scores = {name: [] for name in benzene_structures}
    targets = {
        name: normalize_pi_set(pi_bonds)
        for name, pi_bonds in benzene_structures.items()
    }
    for alt in res.get("alternatives", []):
        alt_pi = normalize_pi_set(alt.get("pi_bonds", []))
        for name, target in targets.items():
            if alt_pi == target:
                weights[name] += float(alt.get("weight", 0.0))
                scores[name].append(float(alt.get("score", 0.0)))
    return weights, scores

# Automatic resonance enumeration with Kekule and Dewar pi choices available.
nbo_benzene_auto = vlx.NboDriver()
nbo_benzene_auto.verbose = False
res_benzene_auto = nbo_benzene_auto.compute(
    mol_benzene,
    bas_benzene,
    scf_benzene.mol_orbs,
    constraints={"allowed_pi_bonds": all_benzene_pi_bonds},
    options=options_benzene,
)
auto_weights, auto_scores = benzene_structure_weight_map(res_benzene_auto)

print("Benzene Kekule/Dewar resonance score weights (HF/def2-SVP)")
print("=" * 104)
print("Automatic enumeration with six ring pi bonds plus three Dewar para pi bonds allowed")
print("-" * 104)
print(f"{'Structure':<34} {'Class':<8} {'Score wt':>10}  {'Score(s)'}")
print("-" * 104)
for name in benzene_structures:
    structure_class = "Kekule" if name.startswith("Kekule") else "Dewar"
    score_text = ", ".join(f"{score:.5f}" for score in auto_scores[name]) or "n/a"
    print(f"{name:<34} {structure_class:<8} {100*auto_weights[name]:>9.2f}%  {score_text}")

print()
print("All enumerated alternatives")
print("-" * 104)
for alt in res_benzene_auto.get("alternatives", []):
    pi_set = normalize_pi_set(alt.get("pi_bonds", []))
    label = "unassigned"
    for name, target in ((name, normalize_pi_set(pi_bonds)) for name, pi_bonds in benzene_structures.items()):
        if pi_set == target:
            label = name
            break
    pi_text = ", ".join(f"{i}-{j}" for i, j in alt.get("pi_bonds", [])) or "none"
    print(
        f"rank={alt.get('rank', 0):>2} "
        f"score_weight={100*alt.get('weight', 0.0):>7.2f}% "
        f"score={alt.get('score', 0.0):>10.5f} "
        f"pi={pi_text:<20} {label}"
    )

print()
print("Forced structures with required_pi_bonds")
print("-" * 104)
forced_results = {}
for name, pi_bonds in benzene_structures.items():
    nbo_forced = vlx.NboDriver()
    nbo_forced.verbose = False
    res_forced = nbo_forced.compute(
        mol_benzene,
        bas_benzene,
        scf_benzene.mol_orbs,
        constraints={"required_pi_bonds": pi_bonds},
        options=options_benzene,
    )
    forced_results[name] = res_forced
    primary = res_forced.get("primary", {})
    structure_class = "Kekule" if name.startswith("Kekule") else "Dewar"
    print(
        f"{name:<34} {structure_class:<8} "
        f"primary score_weight={100*primary.get('weight', 0.0):7.2f}% "
        f"score={primary.get('score', 0.0):10.5f} "
        f"counts={primary.get('counts', {})}"
    )

print("=" * 104)
print("Expectation: the two Kekule structures should be nearly degenerate by score; Dewar structures should be lower-score alternatives.")

                                                                                                                          
                                            Self Consistent Field Driver Setup                                            
                                                                                                                          
                   Wave Function Model             : Spin-Restricted Hartree-Fock                                         
                   Initial Guess Model             : Superposition of Atomic Densities                                    
                   Convergence Accelerator         : Two Level Direct Inversion of Iterative Subspace                     
                   Max. Number of Iterations       : 50                                                                   
                   Max. Number of Error Vectors    : 10                                                                   
                

## 6d. Benzene resonance classes

The same benzene enumeration is grouped into symmetry-equivalent resonance classes. Class weights sum over all member alternatives, so the two Kekule structures are reported as one degenerate class rather than as separate rows.

In [4]:
# Benzene resonance-class summary
resonance_classes = res_benzene_auto.get("resonance_classes", [])
assert resonance_classes

print("Benzene resonance classes from automatic enumeration")
print("=" * 112)
print(f"{'Class':<6} {'Deg.':>4} {'Weight sum':>11} {'Weight range':>23} {'Members':<18} Label")
print("-" * 112)
for record in resonance_classes:
    members = ", ".join(str(rank) for rank in record.get("member_ranks", []))
    weight_range = (
        f"{100.0 * record.get('class_weight_min', 0.0):6.2f}%.."
        f"{100.0 * record.get('class_weight_max', 0.0):6.2f}%"
    )
    print(
        f"{record.get('class_id', ''):<6} "
        f"{record.get('class_degeneracy', 0):>4} "
        f"{100.0 * record.get('class_weight_sum', 0.0):>10.2f}% "
        f"{weight_range:>23} "
        f"{members:<18} "
        f"{record.get('class_label', '')}"
    )

print("\nAlternatives with class IDs")
print("-" * 112)
print(f"{'Rank':>4} {'Class':<6} {'Deg.':>4} {'Score wt':>10} {'Pi bonds':<20} Resonance label")
print("-" * 112)
for alt in res_benzene_auto.get("alternatives", []):
    pi_text = ", ".join(f"{i}-{j}" for i, j in alt.get("pi_bonds", [])) or "none"
    print(
        f"{alt.get('rank', 0):>4} "
        f"{alt.get('resonance_class_id', ''):<6} "
        f"{alt.get('class_degeneracy', 0):>4} "
        f"{100.0 * alt.get('weight', 0.0):>9.2f}% "
        f"{pi_text:<20} "
        f"{alt.get('resonance_label', '')}"
    )

assert any(record.get("class_degeneracy", 0) > 1 for record in resonance_classes)
assert abs(sum(record.get("class_weight_sum", 0.0) for record in resonance_classes) - 1.0) < 1.0e-12
print("=" * 112)
print("PASS: symmetry-equivalent alternatives are grouped into resonance classes with normalized class weights.")

Benzene resonance classes from automatic enumeration
Class  Deg.  Weight sum            Weight range Members            Label
----------------------------------------------------------------------------------------------------------------
R1        2      99.99%         49.99%.. 49.99% 1, 2               pi:1-2,3-4,5-6
R2        3       0.01%          0.00%..  0.00% 3, 4, 5            pi:1-2,3-6,4-5
R3        1       0.00%          0.00%..  0.00% 6                  pi:1-4,2-5,3-6

Alternatives with class IDs
----------------------------------------------------------------------------------------------------------------
Rank Class  Deg.   Score wt Pi bonds             Resonance label
----------------------------------------------------------------------------------------------------------------
   1 R1        2     49.99% 1-6, 2-3, 4-5        pi:1-6,2-3,4-5
   2 R1        2     49.99% 1-2, 3-4, 5-6        pi:1-2,3-4,5-6
   3 R2        3      0.00% 1-4, 2-3, 5-6        pi:1-4,2-3,5-6
   

In [ ]:
# 6e. First NRA/NRT density-fit check
import numpy as np

nbodriver_module = load_local_driver_module(
    "veloxchem.nbodriver",
    "src/pymodule/nbodriver.py",
)
vlx.NboDriver = nbodriver_module.NboDriver

nra_options_benzene = dict(options_benzene)
nra_options_benzene.update({
    "include_nra": True,
    "nra_subspace": "pi",
    "nra_fit_metric": "frobenius",
})

nbo_benzene_nra = vlx.NboDriver()
nbo_benzene_nra.verbose = False
res_benzene_nra = nbo_benzene_nra.compute(
    mol_benzene,
    bas_benzene,
    scf_benzene.mol_orbs,
    constraints={"allowed_pi_bonds": all_benzene_pi_bonds},
    options=nra_options_benzene,
)

nra = res_benzene_nra["nra"]
nra_weights = np.array(nra["weights"])

assert nra["subspace"] == "pi"
assert nra["fit_metric"] == "frobenius"
assert len(nra_weights) == len(res_benzene_nra["alternatives"])
assert np.all(nra_weights >= -1.0e-12)
assert abs(float(np.sum(nra_weights)) - 1.0) < 1.0e-10
assert np.isfinite(nra["residual_norm"])
assert np.isfinite(nra["relative_residual"])

structure_targets = {
    name: normalize_pi_set(pi_bonds)
    for name, pi_bonds in benzene_structures.items()
}
kekule_targets = {
    "Kekule A": normalize_pi_set(benzene_structures["Kekule A (1-2,3-4,5-6)"]),
    "Kekule B": normalize_pi_set(benzene_structures["Kekule B (2-3,4-5,6-1)"]),
}
kekule_nra_weights = {}

print("Benzene first NRA/NRT density-fit check (HF/def2-svp)")
print("=" * 112)
print(
    f"NRA subspace={nra['subspace']}  metric={nra['fit_metric']}  "
    f"residual={nra['residual_norm']:.6e}  relative={nra['relative_residual']:.6e}  "
    f"weight_sum={np.sum(nra_weights):.12f}"
)
if nra.get("warnings"):
    print("Warnings: " + "; ".join(nra["warnings"]))
print("-" * 112)
print(f"{'Rank':>4} {'NRA wt':>10} {'Score wt':>10} {'Score':>11}  {'Pi bonds':<20} {'Label'}")
print("-" * 112)

for structure in nra["structures"]:
    pi_set = normalize_pi_set(structure["pi_bonds"])
    label = "unassigned"
    for name, target in structure_targets.items():
        if pi_set == target:
            label = name
            break
    for name, target in kekule_targets.items():
        if pi_set == target:
            kekule_nra_weights[name] = structure["nra_weight"]

    pi_text = ", ".join(f"{i}-{j}" for i, j in structure["pi_bonds"]) or "none"
    print(
        f"{structure['rank']:>4} "
        f"{100.0 * structure['nra_weight']:>9.3f}% "
        f"{100.0 * structure['score_weight']:>9.3f}% "
        f"{structure['score']:>11.5f}  "
        f"{pi_text:<20} {label}"
    )

assert len(kekule_nra_weights) == 2
assert abs(kekule_nra_weights["Kekule A"] - kekule_nra_weights["Kekule B"]) < 1.0e-8

print("=" * 112)
print("PASS: NRA/NRT result is present, normalized, finite, nonnegative, and symmetric for the two Kekule structures.")

In [ ]:
mol_benzene.show(atom_indices=True)